# Grover's Algorithm

In [1]:
import cirq
import random
import matplotlib.pyplot as plt
import numpy as np

Consider bitstrings of length $n$ and let $x' \in \{0, 1\}^{n}$ be a "marked" bitstring we wish to find. Grover's algorithm takes a black-box oracle implementing a function $f : \{0, 1\}^n \rightarrow \{0, 1\}$ defined by 

$$
f(x) = 1\text{ if } x = x',~~~~ f(x) = 0 \text{ if } x \neq x'
$$ 

to find such a bitstring $x'$. Grover's algorithm uses $O(\sqrt{N}$) operations and $O(N\, \log N$) gates and succeeds with probability $p \geq 2/3$.

Below, we walk through a simple implementation of Grover's algorithm described in [this reference](https://arxiv.org/abs/1804.03719).

<div class="alert alert-block alert-info">
This implementation only supports $n = 2$ (for which one application of the Grover iteration is enough).
</div>

First we define our qubit registers. We use $n = 2$ bits in one register and an additional ancilla qubit for phase kickback.

In [2]:
"""Get qubits to use in the circuit for Grover's algorithm."""
# Number of qubits n.
nqubits = 2

# Get qubit registers.
qubits = cirq.LineQubit.range(nqubits)
ancilla = cirq.NamedQubit("Ancilla")

We now define a generator to yield the operations for the oracle. As discussed in the above reference, the oracle can be implemented by a Toffoli gate if all the bits in $x'$ are $1$. If some bits are $0$, we do an "open control" (control on the $|0\rangle$ state) for these bits. This can be accomplished by flipping every $0$ bit with $X$ gates, performing a Tofolli, then undoing the $X$ gates.

In [3]:
def make_oracle(qubits, ancilla, xprime):
    """Implements the function {f(x) = 1 if x == x', f(x) = 0 if x != x'}."""
    # For x' = (1, 1), the oracle is just a Toffoli gate.
    # For a general x', we negate the zero bits and implement a Toffoli.
    
    # Negate zero bits, if necessary.
    yield (cirq.X(q) for (q, bit) in zip(qubits, xprime) if not bit)
    
    # Do the Toffoli.
    yield (cirq.TOFFOLI(qubits[0], qubits[1], ancilla))
    
    # Negate zero bits, if necessary.
    yield (cirq.X(q) for (q, bit) in zip(qubits, xprime) if not bit)

Now that we have a function to implement the oracle, we can construct a function to implement one round of Grover's iteration.

In [4]:
def grover_iteration(qubits, ancilla, oracle):
    """Performs one round of the Grover iteration."""
    circuit = cirq.Circuit()

    # Create an equal superposition over input qubits.
    circuit.append(cirq.H.on_each(*qubits))
    
    # Put the output qubit in the |-⟩ state.
    circuit.append([cirq.X(ancilla), cirq.H(ancilla)])

    # Query the oracle.
    circuit.append(oracle)

    # Construct Grover operator.
    circuit.append(cirq.H.on_each(*qubits))
    circuit.append(cirq.X.on_each(*qubits))
    circuit.append(cirq.H.on(qubits[1]))
    circuit.append(cirq.CNOT(qubits[0], qubits[1]))
    circuit.append(cirq.H.on(qubits[1]))
    circuit.append(cirq.X.on_each(*qubits))
    circuit.append(cirq.H.on_each(*qubits))

    # Measure the input register.
    circuit.append(cirq.measure(*qubits, key="result"))

    return circuit

We now select the bitstring $x'$ at random.

In [5]:
"""Select a 'marked' bitstring x' at random."""
xprime = [random.randint(0, 1) for _ in range(nqubits)]
print(f"Marked bitstring: {xprime}")

Marked bitstring: [1, 0]


And now create the circuit for Grover's algorithm.

In [6]:
"""Create the circuit for Grover's algorithm."""
# Make oracle (black box)
oracle = make_oracle(qubits, ancilla, xprime)

# Embed the oracle into a quantum circuit implementing Grover's algorithm.
circuit = grover_iteration(qubits, ancilla, oracle)
print("Circuit for Grover's algorithm:")
print(circuit)

Circuit for Grover's algorithm:
0: ─────────H───────@───H───X───────────@───X───H───────M('result')───
                    │                   │               │
1: ─────────H───X───@───X───H───X───H───X───H───X───H───M─────────────
                    │
Ancilla: ───X───H───X─────────────────────────────────────────────────


All that is left is to simulate the circuit and check if the sampled bitstring(s) match with the marked bitstring $x'$.

In [7]:
"""Simulate the circuit for Grover's algorithm and check the output."""
# Helper function.
def bitstring(bits):
    return "".join(str(int(b)) for b in bits)

# Sample from the circuit a couple times.
simulator = cirq.Simulator()
result = simulator.run(circuit, repetitions=10)

# Look at the sampled bitstrings.
frequencies = result.histogram(key="result", fold_func=bitstring)
print('Sampled results:\n{}'.format(frequencies))

# Check if we actually found the secret value.
most_common_bitstring = frequencies.most_common(1)[0][0]
print("\nMost common bitstring: {}".format(most_common_bitstring))
print("Found a match? {}".format(most_common_bitstring == bitstring(xprime)))

Sampled results:
Counter({'10': 10})

Most common bitstring: 10
Found a match? True


We see that we indeed found the marked bitstring $x'$. One can rerun these cells to select a new bitstring $x'$ and check that Grover's algorithm can again find it.

In [8]:
#@title Copyright 2020 The Cirq Developers
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.